In [0]:
%run ../01_bronze/bronze_functions

In [0]:
%run ./silver_functions

In [0]:
load_range("2015-01-12", "2015-01-12", "backfill")

In [0]:
load_silver("2015-01-01", "2015-01-01", "incremental")

In [0]:
load_silver("2015-01-01", "2015-01-01", "incremental")

In [0]:
%sql
SELECT
  SUM(CASE WHEN symbol NOT LIKE '%-%' THEN 1 ELSE 0 END) AS stocks,
  SUM(CASE WHEN symbol LIKE '%-%' THEN 1 ELSE 0 END)     AS futures
FROM workspace.bronze.mkt_summary
WHERE file_trade_date = '2015-01-12'    

In [0]:
%sql
DESCRIBE TABLE workspace.silver.daily_prices

In [0]:
%sql
SELECT symbol, trade_date, COUNT(*) AS c
FROM workspace.silver.daily_prices
GROUP BY symbol, trade_date HAVING COUNT(*) > 1

In [0]:
syms = [r.symbol for r in spark.sql("""
    SELECT symbol FROM workspace.silver.daily_prices
    WHERE trade_date = '2015-01-12' AND volume > 0 ORDER BY symbol LIMIT 2""").collect()]
upd_sym, drift_sym = syms
print("Update test:", upd_sym, "| Drift test:", drift_sym)

def show_rows():
    display(spark.sql(f"""
        SELECT symbol, close, volume, batch_id, load_timestamp, updated_timestamp
        FROM workspace.silver.daily_prices
        WHERE trade_date = '2015-01-12' AND symbol IN ('{upd_sym}', '{drift_sym}')"""))

show_rows()

In [0]:
spark.sql(f"""UPDATE workspace.bronze.mkt_summary
              SET volume_raw = CAST(CAST(volume_raw AS BIGINT) + 100 AS STRING)
              WHERE file_trade_date = '2015-01-12' AND trim(symbol) = '{upd_sym}'""")

spark.sql(f"""UPDATE workspace.bronze.mkt_summary
              SET close_raw = 'abc'
              WHERE file_trade_date = '2015-01-12' AND trim(symbol) = '{drift_sym}'""")
print("Bronze modified")

In [0]:
load_silver("2015-01-12", "2015-01-12", "backfill")

In [0]:
show_rows()
display(spark.sql("""
    SELECT reason, raw_record, batch_id FROM workspace.ops.quarantine_mkt_summary
    WHERE layer = 'silver' AND file_trade_date = '2015-01-12'"""))

In [0]:
load_range("2015-01-12", "2015-01-12", "backfill")   # Bronze reloads from the real file
load_silver("2015-01-12", "2015-01-12", "backfill")  # Silver fixes itself
show_rows()

In [0]:
%sql
SELECT batch_id, load_type, parameter, status, rows_inserted, rows_updated, rows_quarantined
FROM workspace.ops.pipeline_execution_logs
WHERE layer = 'bronze_to_silver' ORDER BY start_time